# Retail-IQ: Exploratory Data Analysis

This notebook has been refactored to use the modular `retail_iq` package for better maintainability and professional standards.

## FR-01 & FR-02: Data Loading and Preprocessing

In [2]:
import os
from retail_iq import load_raw_data, preprocess_dates, clean_oil_prices, merge_datasets, detect_outliers_iqr, LOG_DIR

# 1. Load Data
train, test, stores, oil, holidays, transactions = load_raw_data()
print("Data loaded successfully.")

# 2. Preprocess Dates
train, test, stores, oil, holidays, transactions = preprocess_dates([train, test, stores, oil, holidays, transactions])

# 3. Clean Oil Prices
oil = clean_oil_prices(oil)

# 4. Merge Datasets
df = merge_datasets(train, stores, oil, holidays, transactions)

# 5. Detect Outliers
df = detect_outliers_iqr(df)

print(f"Preprocessed dataset shape: {df.shape}")

# Log progress
with open(LOG_DIR / "preprocessing_log.txt", "w") as f:
    f.write(f"Final row count after preprocessing: {len(df)}\n")

Data loaded successfully.
Preprocessed dataset shape: (3000888, 18)


C:\Users\mypc\Downloads\Retail-IQ\src\retail_iq\preprocessing.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(['store_nbr','family'], group_keys=False).apply(get_outliers)


## FR-03: Feature Engineering

In [3]:
from retail_iq import FastFeatureEngineer

fe = FastFeatureEngineer(
    df=train, # Using original train as in original notebook logic
    transactions=transactions,
    oil_price=oil,
    holidays=holidays,
    store_meta=stores
)

train_features = (
    fe.add_temporal_features()
      .add_lag_and_rolling()
      .add_onpromotion_features()
      .add_macroeconomic_features()
      .add_transaction_features()
      .add_store_metadata()
      .add_cannibalization_features()
      .transform()
)

train_features = train_features.dropna()
print(f"Features engineered. Shape: {train_features.shape}")

Features engineered. Shape: (1567564, 35)


## FR-04: Visualization

In [4]:
from retail_iq import plot_ts_decomposition, plot_correlation_heatmap, plot_sales_distribution

# 1. TS Decomposition samples
samples = train_features.groupby(['store_nbr','family']).size().sample(3, random_state=42).index
for store_nbr, family in samples:
    path = plot_ts_decomposition(train_features, store_nbr, family)
    print(f"Saved decomposition plot to {path}")

# 2. Correlation Heatmap
path = plot_correlation_heatmap(train_features)
print(f"Saved heatmap to {path}")

# 3. Sales Distribution
path = plot_sales_distribution(train_features)
print(f"Saved distribution plot to {path}")

Saved decomposition plot to C:\Users\mypc\Downloads\Retail-IQ\outputs\plots\ts_decompose_store25_familySEAFOOD.png
Saved decomposition plot to C:\Users\mypc\Downloads\Retail-IQ\outputs\plots\ts_decompose_store45_familyLADIESWEAR.png
Saved decomposition plot to C:\Users\mypc\Downloads\Retail-IQ\outputs\plots\ts_decompose_store8_familyBEAUTY.png
Saved heatmap to C:\Users\mypc\Downloads\Retail-IQ\outputs\plots\correlation_heatmap.png
Saved distribution plot to C:\Users\mypc\Downloads\Retail-IQ\outputs\plots\sales_distribution.png
